# 01 — Embeddings & the Resume Parser

Prototype two things here, then move the working code into `src/`:

1. **Resume parser** — load a resume with `src.parsing.loader.load_text`, send it to the LLM with a strict prompt, get back clean JSON (name, skills, experience, education, target_role). Validate the JSON. Move it to `src/parsing/resume_parser.py`.
2. **Embeddings** — embed a few short texts with the Gemini embedding model, compute cosine similarity between related vs unrelated sentences, and confirm similar meaning gives a higher score. Move the embedding call to `src/search/embed.py`.

Load your API key from `.env` first (`python-dotenv`).

In [12]:
# ============================================================
# SMART HIRE - NOTEBOOK 01
# Resume Parser using TypedDict / Structured Output
# ============================================================

import sys
import os
import json
from pathlib import Path
from typing import TypedDict, List

# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

# Add SmartHire-GenAI to Python path so "src" can be imported
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT)


# ------------------------------------------------------------
# 2. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

from dotenv import load_dotenv
from google import genai
from google.genai import types

from src.parsing.loader import load_text


# ------------------------------------------------------------
# 3. LOAD API KEY FROM .env.example
# ------------------------------------------------------------

env_file = PROJECT_ROOT / ".env.example"

load_dotenv(env_file)

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found in .env.example"
    )

print("API key loaded successfully.")


# ------------------------------------------------------------
# 4. CONNECT TO GEMINI
# ------------------------------------------------------------

client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini client connected.")


# ------------------------------------------------------------
# 5. DEFINE TYPEDDICT STRUCTURE
# ------------------------------------------------------------

class Education(TypedDict):
    degree: str
    institute: str
    year: str
    score: str


class Experience(TypedDict):
    company: str
    role: str
    duration: str
    highlights: List[str]


class Resume(TypedDict):
    name: str
    skills: List[str]
    experience: List[Experience]
    education: List[Education]
    target_role: str


# ------------------------------------------------------------
# 6. STRICT AI PROMPT
# ------------------------------------------------------------

SYSTEM_PROMPT = """
You are an expert resume parser.

Extract information ONLY from the resume provided.

Do NOT invent, guess, or assume information.

Return the information using exactly these fields:

name
skills
experience
education
target_role

For experience, extract:

company
role
duration
highlights

For education, extract:

degree
institute
year
score

Rules:

1. Extract only information explicitly present in the resume.
2. Do not invent missing information.
3. If a string value is missing, return an empty string.
4. If a list value is missing, return an empty list.
5. skills must contain the skills explicitly mentioned.
6. experience must contain the candidate's work experience.
7. education must contain the candidate's education.
8. target_role should be based only on the candidate's resume.
9. Return only structured JSON matching the Resume schema.
"""


# ------------------------------------------------------------
# 7. RESUME PARSER USING TYPEDDICT
# ------------------------------------------------------------

def parse_resume(text: str) -> dict:

    response = client.models.generate_content(
        model=MODEL,
        contents=f"""
{SYSTEM_PROMPT}

RESUME:
{text}
""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Resume,
            temperature=0.2,
            max_output_tokens=2048
        )
    )

    return json.loads(response.text)


# ------------------------------------------------------------
# 8. FIND ALL RESUMES
# ------------------------------------------------------------

resume_folder = PROJECT_ROOT / "data" / "resumes"

if not resume_folder.exists():
    raise FileNotFoundError(
        f"Resume folder not found: {resume_folder}"
    )

resume_files = [
    file
    for file in resume_folder.iterdir()
    if file.suffix.lower() in [".pdf", ".docx", ".txt", ".md"]
]

print("\nResumes found:", len(resume_files))

for file in resume_files:
    print("-", file.name)


# ------------------------------------------------------------
# 9. LOAD AND PARSE ALL THREE RESUMES
# ------------------------------------------------------------

parsed_resumes = {}

for file in resume_files:

    print("\n" + "=" * 70)
    print("PROCESSING:", file.name)
    print("=" * 70)

    try:

        # Read resume using your existing loader.py
        resume_text = load_text(file)

        print("Resume loaded successfully.")
        print("Characters extracted:", len(resume_text))

        # Parse using TypedDict structured output
        parsed_data = parse_resume(resume_text)

        parsed_resumes[file.name] = parsed_data

        print("\nSTRUCTURED OUTPUT:")
        print(
            json.dumps(
                parsed_data,
                indent=2,
                ensure_ascii=False
            )
        )

    except Exception as e:

        print("\nERROR:")
        print(e)


# ------------------------------------------------------------
# 10. VALIDATE REQUIRED PROJECT HEADINGS
# ------------------------------------------------------------

required_fields = {
    "name",
    "skills",
    "experience",
    "education",
    "target_role"
}

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

for filename, data in parsed_resumes.items():

    missing_fields = required_fields - set(data.keys())

    if missing_fields:
        print(
            f"❌ {filename} missing fields: "
            f"{missing_fields}"
        )
    else:
        print(
            f"✅ {filename} contains all required fields."
        )


# ------------------------------------------------------------
# 11. DISPLAY RESUME SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESUME SUMMARY")
print("=" * 70)

for filename, data in parsed_resumes.items():

    print("\nResume:", filename)
    print("Name:", data.get("name"))
    print("Skills:", data.get("skills"))
    print("Experience:", data.get("experience"))
    print("Education:", data.get("education"))
    print("Target Role:", data.get("target_role"))


print("\n" + "=" * 70)
print("RESUME PARSER COMPLETED")
print("=" * 70)

Project folder: c:\Users\jeesh\Smart_hire_Gen_AI_June
API key loaded successfully.
Gemini client connected.

Resumes found: 3
- 01_Bhawana_Aggarwal_Data_Science.pdf
- Ananya_Sharma_Resume.pdf
- Rahul_Verma_Resume.pdf

PROCESSING: 01_Bhawana_Aggarwal_Data_Science.pdf
Resume loaded successfully.
Characters extracted: 1680

STRUCTURED OUTPUT:
{
  "name": "BHAWANA AGGARWAL",
  "skills": [
    "Python",
    "C",
    "NumPy",
    "Pandas",
    "Seaborn",
    "Matplotlib",
    "Cufflinks",
    "Machine Learning",
    "KNN",
    "Decision Tree",
    "Linear & Logistic Regression",
    "SVM",
    "K-Means",
    "Deep Learning",
    "Neural Networks",
    "TensorFlow",
    "Keras",
    "RNN/LSTM",
    "NLP",
    "NER",
    "Regex",
    "SQL",
    "Oracle",
    "GCP",
    "Linux",
    "Windows"
  ],
  "experience": [
    {
      "company": "Wipro Technologies",
      "role": "Data Science / Machine Learning",
      "duration": "2 years",
      "highlights": [
        "Developed Python programs an

In [6]:
# ============================================================
# SMART HIRE - NOTEBOOK 01
# Resume → Job Semantic Matching
# ============================================================
# ============================================================
# SMART HIRE - NOTEBOOK 01
# Resume → 22,000 Job Semantic Matching
# ============================================================

import sys
import os
import time
from pathlib import Path

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

current = Path.cwd()

for folder in [current] + list(current.parents):
    if (
        (folder / "src").is_dir()
        and (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ):
        PROJECT_ROOT = folder
        break
else:
    raise FileNotFoundError(
        "Could not find SmartHire project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT)

# ------------------------------------------------------------
# 3. IMPORT PROJECT FUNCTIONS
# ------------------------------------------------------------

from src.parsing.loader import load_text

from src.search.embed import (
    embed_text,
    embed_texts,
    cosine_similarity,
    EMBEDDING_DIMENSION
)

print("Embedding dimension:", EMBEDDING_DIMENSION)


# ------------------------------------------------------------
# 4. FIND JOB DATASET
# ------------------------------------------------------------

jobs_folder = PROJECT_ROOT / "data" / "jobs"

if not jobs_folder.exists():
    raise FileNotFoundError(
        f"Jobs folder not found: {jobs_folder}"
    )


job_files = list(
    jobs_folder.glob("*.csv")
)

if not job_files:
    raise FileNotFoundError(
        f"No CSV job dataset found in {jobs_folder}"
    )


print("\nJob files found:")

for file in job_files:
    print("-", file.name)


# ------------------------------------------------------------
# 5. LOAD JOB DATASET
# ------------------------------------------------------------

job_file = job_files[0]

jobs_df = pd.read_csv(
    job_file,
    low_memory=False
)

print("\nJob dataset loaded.")
print("Number of jobs:", len(jobs_df))

print("\nColumns:")
print(jobs_df.columns.tolist())


# ------------------------------------------------------------
# 6. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = {
    "jobtitle",
    "jobdescription"
}

missing_columns = (
    required_columns
    - set(jobs_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 7. CLEAN JOB DATA
# ------------------------------------------------------------

jobs_df = jobs_df.fillna("")

jobs_df["jobtitle"] = (
    jobs_df["jobtitle"]
    .astype(str)
)

jobs_df["jobdescription"] = (
    jobs_df["jobdescription"]
    .astype(str)
)


# ------------------------------------------------------------
# 8. CREATE TEXT REPRESENTATION FOR EACH JOB
# ------------------------------------------------------------

jobs_df["job_text"] = (
    "Job Title: "
    + jobs_df["jobtitle"]
    + "\nJob Description: "
    + jobs_df["jobdescription"]
)


print(
    "\nPrepared",
    len(jobs_df),
    "jobs for embedding."
)


# ------------------------------------------------------------
# 9. FIND RESUMES
# ------------------------------------------------------------

resume_folder = (
    PROJECT_ROOT
    / "data"
    / "resumes"
)

if not resume_folder.exists():
    raise FileNotFoundError(
        f"Resume folder not found: {resume_folder}"
    )


resume_files = [
    file
    for file in resume_folder.iterdir()
    if file.suffix.lower() in [
        ".pdf",
        ".docx",
        ".txt",
        ".md"
    ]
]


if not resume_files:
    raise FileNotFoundError(
        "No resumes found."
    )


print("\nResumes available:")

for i, file in enumerate(resume_files):
    print(
        f"{i}: {file.name}"
    )


# ------------------------------------------------------------
# 10. SELECT RESUME
# ------------------------------------------------------------

# Change this number if you want another resume.
RESUME_INDEX = 0

resume_file = resume_files[
    RESUME_INDEX
]

print(
    "\nSelected resume:",
    resume_file.name
)


# ------------------------------------------------------------
# 11. LOAD RESUME
# ------------------------------------------------------------

resume_text = load_text(
    resume_file
)

if not resume_text.strip():
    raise ValueError(
        "Resume text could not be extracted."
    )


print(
    "Resume loaded successfully."
)

print(
    "Characters extracted:",
    len(resume_text)
)
# ============================================================
# 12. CREATE RESUME EMBEDDING
# ============================================================

print("\nCreating resume embedding...")

resume_embedding = embed_text(resume_text)

print("Resume embedding shape:", resume_embedding.shape)

if resume_embedding.shape != (384,):
    raise ValueError(
        f"Expected resume embedding shape (384,), "
        f"but got {resume_embedding.shape}"
    )

print("Resume embedding created successfully.")

# ============================================================
# 13. CREATE EMBEDDINGS FOR ALL JOBS
# ============================================================

print("\n" + "=" * 80)
print("CREATING EMBEDDINGS FOR ALL JOBS")
print("=" * 80)

total_jobs = len(jobs_df)

print("Total jobs:", total_jobs)
print("Embedding dimension: 384")

# Create job text list
job_texts = jobs_df["job_text"].tolist()

# Embed all jobs
job_embeddings = embed_texts(
    job_texts
)

# Convert to NumPy array
job_embeddings = np.asarray(
    job_embeddings,
    dtype=np.float32
)

print("\nJob embeddings created.")
print("Job embeddings shape:", job_embeddings.shape)

# Validate dimensions
expected_shape = (total_jobs, 384)

if job_embeddings.shape != expected_shape:
    raise ValueError(
        f"Expected shape {expected_shape}, "
        f"but got {job_embeddings.shape}"
    )

print("✅ All job embeddings have 384 dimensions.")
# ============================================================
# 14. SAVE JOB EMBEDDINGS
# ============================================================

embeddings_folder = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
)

embeddings_folder.mkdir(
    parents=True,
    exist_ok=True
)

embeddings_file = (
    embeddings_folder
    / "job_embeddings_bge_small.npy"
)

np.save(
    embeddings_file,
    job_embeddings
)

print("\n✅ Job embeddings saved successfully.")
print("Saved file:", embeddings_file)

# ============================================================
# 15. CALCULATE COSINE SIMILARITY
# ============================================================

print("\nCalculating cosine similarity...")

scores = (
    job_embeddings
    @ resume_embedding
)

jobs_df["similarity"] = scores

print("✅ Similarity calculated for all jobs.")

print(
    "Number of similarity scores:",
    len(scores)
)

# ============================================================
# 16. RANK ALL JOBS
# ============================================================

ranked_jobs = (
    jobs_df
    .sort_values(
        by="similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

print("✅ All jobs ranked successfully.")

# ============================================================
# 17. DISPLAY TOP MATCHING JOBS
# ============================================================

TOP_N = 10

print(
    "\n"
    + "=" * 80
)

print("TOP 10 MATCHING JOBS")

print("=" * 80)

for i, row in ranked_jobs.head(TOP_N).iterrows():

    print(f"\nRank {i + 1}")

    print("Job ID:", row["jobid"])

    print("Job Title:", row["jobtitle"])

    print(
        "Cosine Similarity:",
        round(
            float(row["similarity"]),
            4
        )
    )

    print(
        "Location:",
        row.get(
            "joblocation_address",
            ""
        )
    )

    print(
        "Description:",
        row["jobdescription"][:500],
        "..."
    )
# ============================================================
# 18. RANK ALL 22,000 JOBS
# ============================================================

ranked_jobs = (
    jobs_df
    .sort_values(
        by="similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

print("✅ All jobs ranked successfully.")

print(
    "Total ranked jobs:",
    len(ranked_jobs)
)

# ============================================================
# 19. DISPLAY TOP MATCHING JOBS
# ============================================================

TOP_N = 10

print(
    "\n" + "=" * 80
)

print("TOP 10 MATCHING JOBS")

print("=" * 80)

for i, row in ranked_jobs.head(TOP_N).iterrows():

    print(f"\nRank {i + 1}")

    print("Job ID:", row["jobid"])

    print("Job Title:", row["jobtitle"])

    print(
        "Cosine Similarity:",
        round(
            float(row["similarity"]),
            4
        )
    )

    print(
        "Location:",
        row.get(
            "joblocation_address",
            ""
        )
    )

    print(
        "Description:",
        row["jobdescription"][:500],
        "..."
    )



# ============================================================
# 20. FINAL RESULT TABLE
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FINAL JOB RANKING"
)

print(
    "=" * 80
)


result_columns = [
    "jobid",
    "jobtitle",
    "similarity"
]

available_columns = [
    column
    for column in result_columns
    if column in ranked_jobs.columns
]


print(
    ranked_jobs[
        available_columns
    ]
    .head(TOP_N)
    .to_string(index=False)
)


print(
    "\n"
    + "=" * 80
)

print(
    "JOB MATCHING COMPLETED"
)

print(
    "=" * 80
)

Project folder: c:\Users\jeesh\Smart_hire_Gen_AI_June
Embedding dimension: 384

Job files found:
- naukri_com-job_sample.csv

Job dataset loaded.
Number of jobs: 22000

Columns:
['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']

Prepared 22000 jobs for embedding.

Resumes available:
0: 01_Bhawana_Aggarwal_Data_Science.pdf
1: Ananya_Sharma_Resume.pdf
2: Rahul_Verma_Resume.pdf

Selected resume: 01_Bhawana_Aggarwal_Data_Science.pdf
Resume loaded successfully.
Characters extracted: 1680

Creating resume embedding...
Resume embedding shape: (384,)
Resume embedding created successfully.

CREATING EMBEDDINGS FOR ALL JOBS
Total jobs: 22000
Embedding dimension: 384


Batches: 100%|██████████| 688/688 [46:52<00:00,  4.09s/it] 



Job embeddings created.
Job embeddings shape: (22000, 384)
✅ All job embeddings have 384 dimensions.

✅ Job embeddings saved successfully.
Saved file: c:\Users\jeesh\Smart_hire_Gen_AI_June\data\embeddings\job_embeddings_bge_small.npy

Calculating cosine similarity...
✅ Similarity calculated for all jobs.
Number of similarity scores: 22000
✅ All jobs ranked successfully.

TOP 10 MATCHING JOBS

Rank 1
Job ID: 220316001622
Job Title: Data Scientist - Bangalore
Cosine Similarity: 0.7829
Location: Bengaluru/Bangalore
Description: Job Description   Send me Jobs like this Dear Candidates, We have an urgent requirement for a Data Scientist Role at Wipro BPS Bangalore. Please go through the Job Description below and in case you are interested then please share your resume at Rahul.kedia@Wipro.com Work Location: Wipro BPS, Sarjapur Office, Bangalore Shift Timings: 11:00 AM - 8:30 PM (Fixed), Monday - Friday Transport : Only one way to be provided The Position: The Data Scientist will be a team 